# Análise de Acessibilidade

**AUTORIA:** [REDE MOB](https://www.redemob.com.br/)

Análise do padrão de viagens de um município com base em dados de pesquisa de origem e destino. As linhas de código abaixo têm como objeto o município de Belo Horizonte, mas serve para qualquer outro, desde que os dados de entrada estejam no layout padrão da Plataforma.

**PANORAMA:**
- #TODO: Listar as análises aqui contidas

**MAIS INFORMAÇÕES:**
- [Layout da Plataforma]
- [Sumário dos Dados Disponíveis]
- *Lorem ipsum: Conteúdo do MOB de interesse, técnico ou de divulgação*

**LINKS DE INTERESSE:**
- links para materiais técnicos e acadêmicos gerais de referência a respeito do conteúdo abordado



# Introdução

Este script foi concebido em caráter de tutorial, tomando como exemplo o município de Belo Horizonte/MG, de forma que as considerações e discussões aqui contidas foram tecidas no contexto dessa municipalidade. Com efeito, procurou-se, na medida do possível, deixar o texto abrangente a ponto de orientar as análises de outros municípios. Ou seja, este script acaba constituindo um apoio para o diagnóstico de outras localidade, na medida em que houve um esforço de trazer técnicas e elementos norteadores para contribuir para o diagnóstico de outros locais. Com efeito, ao alterar os parâmetros de entrada, conforme demonstrado logo abaixo, podem ser gerados mapas e gráficos de territórios distintos. Nesse caso, as considerações originalmente tecidas para Belo Horizonte podem não se aplicar completamente, mas, elas ainda devem fornecer insumos para a interpretação de resultados de outros locais.

# Instruções

Este script precisa de dois grupos de dados:
1. Dados de origem e destino #TODO: explicar como obter
2. Malhas territoriais de zonas de tráfego

Essses dados devem ser atribuídos às variáveis abaixo. Em seguida, deve-se rodar todo o script e os resultados estarão ao final.

# Backend

In [1]:
import datetime as dt
import os
import pathlib
from pathlib import Path
import warnings


import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import partridge as ptg
import r5py
import seaborn as sns
from tqdm.auto import tqdm

In [2]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.float_format = '{:,.2f}'.format

TQDM_BAR_FORMAT = (
    "{desc:<12}{percentage:3.0f}%|{bar}| "
    "{n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]"
)

In [3]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

# Paths

In [4]:
out_folder = Path(os.environ.get('OUT_FOLDER', './outputs'))
db_folder = Path(os.environ.get('DB_FOLDER', './database'))

osm_pbf_path = out_folder / "B/bh.osm.pbf"
dem_path = out_folder / 'A/dem_bh.tiff'
gtfs_root = out_folder / 'B/corrected_gtfs'
centroids_path = out_folder / 'A/pop_weighted_centroids.parquet'
output_base_dir = out_folder / "B/travel_times/transit"

output_base_dir.mkdir(parents=True, exist_ok=True)

print(f"Using output folder: {out_folder}")

Using output folder: C:\Users\brand\OneDrive\Documentos\Coppe\thesis\outputs


# Helpers

In [5]:
def find_date_ordered_gtfs_files(gtfs_dir: Path) -> list[Path]:
    """Finds and sorts GTFS zip files by date encoded in filename."""
    files = list(gtfs_dir.glob("*.zip"))
    if not files:
        warnings.warn(f"No GTFS .zip files found in {gtfs_dir}")
        return []

    def get_date_from_filename(f: Path) -> dt.datetime:
        """Extracts date assuming 'prefix_DD-MM-YYYY.zip' format."""
        try:
            # Adjust splitting/parsing based on actual filename format
            date_str = f.stem.split("_")[-1]
            return dt.datetime.strptime(date_str, "%d-%m-%Y")
        except (IndexError, ValueError):
            warnings.warn(f"Could not parse date from {f.name}.")
            # Assign minimal date to sort unparseable names first
            return dt.datetime.min

    return sorted(files, key=get_date_from_filename)


def get_departure_datetime(
    analysis_date: dt.date, hour: int = 6, minute: int = 0
) -> dt.datetime:
    """Combines a date with a specific time (default 06:00 AM)."""
    if not isinstance(analysis_date, dt.date):
        # Keep basic type check as it's crucial for combine()
        raise TypeError("analysis_date must be a datetime.date object.")
    return dt.datetime.combine(analysis_date, dt.time(hour, minute))

In [6]:
def process_single_gtfs(
    osm_path: Path,
    dem_path_or_none: Path | None,
    gtfs_path: Path,
    origins_subset_gdf: gpd.GeoDataFrame,
    output_dir: Path,
    departure_dt: dt.datetime,
    snap_dist: float,
    aperture_level: int
) -> tuple[str, bool]:
    """
    Builds network, computes travel matrix, saves for one GTFS file.
    Returns (gtfs_stem, success_status) for logging.
    Designed for parallel execution.
    """
    gtfs_stem = gtfs_path.stem
    try:
        transport_network = r5py.TransportNetwork(
            osm_pbf=osm_path,
            gtfs=gtfs_path,
            elevation_model=dem_path_or_none,
        )

        transit_modes = [
            r5py.TransportMode.TRANSIT,
            r5py.TransportMode.WALK,
        ]
        matrix_computer = r5py.TravelTimeMatrix(
            transport_network=transport_network,
            origins=origins_subset_gdf,
            departure=departure_dt,
            departure_time_window=dt.timedelta(hours=2),
            max_time=dt.timedelta(minutes=120),
            max_public_transport_rides=2, # Max 2 transfers allowed
            transport_modes=transit_modes,
            snap_to_network=snap_dist,
        )

        parquet_file = output_dir / f"{gtfs_stem}.parquet"
        matrix_computer.to_parquet(parquet_file, index=False)
        return gtfs_stem, True  # Success

    except Exception as e:
        tqdm.write( # Log errors concisely during parallel runs
            f"🤚 Error (A{aperture_level}, {gtfs_stem}): {type(e).__name__}"
        )
        return gtfs_stem, False # Failure

# Data Loading

In [7]:
# === Load Centroids ===

centroids_gdf = gpd.read_parquet(centroids_path)

ORIGINS_DESTINATIONS = (
    centroids_gdf
    .rename(columns={'hex_id': 'id'}) # Adjust column name if needed
    .reindex(columns=['id', 'aperture', 'geometry'])
    .to_crs(4326) # WGS84
    .set_index('id', drop=False)
)
print(f"Loaded {len(ORIGINS_DESTINATIONS)} origin/destination points.")


# === Check for DEM file ===
dem_input_path: Path | None = None
if dem_path.exists():
    dem_input_path = dem_path
    print(f"Using DEM file: {dem_path.name}")
else:
    warnings.warn(f"DEM file not found: {dem_path}. Proceeding without DEM.")


# === Pre-calculate Busiest Dates for GTFS ===
print("Pre-calculating busiest dates for GTFS feeds...")
all_gtfs_files = find_date_ordered_gtfs_files(gtfs_root)
busiest_dates: dict[Path, dt.date | None] = {}

with tqdm(
    all_gtfs_files, desc="GTFS Dates", bar_format=TQDM_BAR_FORMAT
) as date_bar:
    for gtfs_file in date_bar:
        try:
            date, _ = ptg.read_busiest_date(gtfs_file)
            busiest_dates[gtfs_file] = date
        except Exception as e:
            tqdm.write(f"🤚 Error reading date for {gtfs_file.stem}: {e}")
            busiest_dates[gtfs_file] = None # Mark as unusable


# Filter out GTFS files where date calculation failed
valid_gtfs_tasks = [
    (path, date) for path, date in busiest_dates.items() if date
]
valid_gtfs_count = len(valid_gtfs_tasks)
total_gtfs_count = len(all_gtfs_files)

if total_gtfs_count > 0:
    print(
        f"Found valid dates for {valid_gtfs_count}/{total_gtfs_count} GTFS."
    )
    if valid_gtfs_count == 0:
        warnings.warn("Cannot proceed without valid GTFS dates.")
else:
    warnings.warn("No GTFS files found to process.")

Loaded 166447 origin/destination points.
Using DEM file: dem_bh.tiff
Pre-calculating busiest dates for GTFS feeds...


GTFS Dates    0%|          | 0/21 [00:00<?, ?it/s]

Found valid dates for 21/21 GTFS.


# 4. Calculate Travel Time Matrices (Parallel)

In [8]:
grouped_origins = ORIGINS_DESTINATIONS.groupby("aperture")
num_apertures = ORIGINS_DESTINATIONS['aperture'].nunique()

In [9]:
# Use all available CPU cores for parallel processing
n_jobs = -1

In [ ]:
if valid_gtfs_count > 0: # Proceed only if there are valid GTFS files
    # === Outer loop: Apertures ===
    with tqdm(
        grouped_origins,
        desc="Apertures",
        total=num_apertures,
        bar_format=TQDM_BAR_FORMAT,
        colour="#AA4499" # Purple
    ) as aperture_bar:
        for aperture, origins_subset in aperture_bar:
            # Skip aperture 9 if needed
            if aperture == 9:
                tqdm.write(f"⏭️ Skipping Aperture {aperture}")
                continue
            aperture_bar.set_postfix_str(f"Res {aperture}")

            # Calculate snap distance once per aperture
            snap_dist = np.ceil(
                h3.average_hexagon_edge_length(aperture, unit="m")
            )
            # Define and create output directory
            aperture_output_dir = output_base_dir / f"aperture_{aperture}"
            aperture_output_dir.mkdir(parents=True, exist_ok=True)

            print(
                f"  Processing {len(valid_gtfs_tasks)} GTFS for A{aperture}..."
            )
            success_count = 0
            fail_count = 0

            # === Inner loop: GTFS files (Sequential) ===
            with tqdm(
                valid_gtfs_tasks,
                desc=f"  GTFS (A{aperture})",
                bar_format=TQDM_BAR_FORMAT,
                colour="#44AA99", # Teal
                leave=False # Keep inner bar contained
            ) as gtfs_bar:
                for gtfs_path, busiest_date in gtfs_bar:
                    gtfs_stem = gtfs_path.stem
                    gtfs_bar.set_postfix_str(f"{gtfs_stem}")
                    departure_dt = get_departure_datetime(busiest_date)
                    output_path = aperture_output_dir / f"{gtfs_stem}.parquet"

                    # Optional: Skip if output exists
                    # if output_path.exists(): continue

                    try:
                        # Build network for this specific GTFS
                        transport_network = r5py.TransportNetwork(
                            osm_pbf=osm_pbf_path,
                            gtfs=gtfs_path,
                            elevation_model=dem_input_path,
                        )

                        transit_modes = [
                            r5py.TransportMode.TRANSIT,
                            r5py.TransportMode.WALK,
                        ]
                        matrix_computer = r5py.TravelTimeMatrix(
                            transport_network=transport_network,
                            origins=origins_subset, # GDF subset
                            departure=departure_dt,
                            departure_time_window=dt.timedelta(hours=2),
                            max_time=dt.timedelta(minutes=120),
                            max_public_transport_rides=2,
                            transport_modes=transit_modes,
                            snap_to_network=snap_dist,
                        )

                        # Save results
                        matrix_computer.to_parquet(output_path, index=False)
                        success_count += 1

                    except Exception as e:
                        # Log errors concisely
                        tqdm.write(
                           f"🤚 Error (A{aperture}, {gtfs_stem}): "
                           f"{type(e).__name__}"
                        )
                        fail_count += 1
                        continue # Move to the next GTFS file

            # --- Log summary for this aperture ---
            tqdm.write(
                f"  ✅ A{aperture}: Processed {success_count} GTFS files "
                f"({fail_count} failed)."
            )

    print("\n--- All Travel Time Calculations Complete ---")
else:
    print("❌ No valid GTFS tasks to process. Calculations skipped.")

# Parent Data to Children

## Routing

In [ ]:
ORIGINS_DESTINATIONS = (
    centroids
    .rename(columns={'hex_id': 'id'})
    .to_crs(31983) # just in case...
    .reindex(columns=['id', 'aperture', 'geometry'])
    .to_crs(4326)
    )

In [ ]:
output_pbf = out_folder / "B/bh.osm.pbf"
dem_path = out_folder / 'A/dem_bh.tiff'
gtfs_root = out_folder / 'B/corrected_gtfs'

In [ ]:
def find_date_ordered_gtfs_files(gtfs_root_path: Path) -> list[Path]:
    """Finds and sorts GTFS zip files by date in filename."""
    # Example assumes filename like 'gtfs_prefix_DD-MM-YYYY.zip'
    files = list(gtfs_root_path.glob("*.zip"))
    if not files:
        warnings.warn(f"No GTFS .zip files found in {gtfs_root_path}")
        return []

    def get_date_from_filename(f: Path) -> dt.datetime:
        try:
            # Adjust splitting/parsing based on actual filename format
            date_str = f.stem.split("_")[-1]
            return dt.datetime.strptime(date_str, "%d-%m-%Y")
        except (IndexError, ValueError):
            warnings.warn(f"Could not parse date from {f.name}, skipping.")
            return dt.datetime.min # Put unparseable files first

    return sorted(files, key=get_date_from_filename)

In [ ]:
def get_departure_datetime(
    analysis_date: dt.date, hour: int = 6, minute: int = 0
) -> dt.datetime:
    """Combines a date with a specific time for departure."""
    return dt.datetime.combine(analysis_date, dt.time(hour, minute))

In [ ]:
gtfs_files_list = find_date_ordered_gtfs_files(gtfs_root)
busiest_dates: dict[Path, dt.date | None] = {}

with tqdm(
    gtfs_files_list, desc="GTFS Dates", bar_format=TQDM_BAR_FORMAT
) as date_bar:
    for gtfs_path in date_bar:
        try:
            date, _ = ptg.read_busiest_date(gtfs_path)
            busiest_dates[gtfs_path] = date
        except Exception as e:
            tqdm.write(
                f"🤚 Error reading busiest date for {gtfs_path.stem}: {e}."
            )
            busiest_dates[gtfs_path] = None # Mark as None

valid_gtfs_count = sum(1 for d in busiest_dates.values() if d is not None)
print(f"Found valid busiest dates for {valid_gtfs_count} GTFS files.")

In [ ]:
with tqdm(
    ORIGINS_DESTINATIONS.groupby("aperture"),
    desc="Apertures",
    bar_format=TQDM_BAR_FORMAT,
    colour="#AA4499"
    ) as outer_bar:
    for aperture, data in outer_bar:
        if aperture == 9:
            continue
        outer_bar.set_postfix_str(f"Resolution {aperture}")

        outpath = out_folder / "B"
        aperture_dir = outpath / f"travel_times/transit/aperture_{aperture}"
        aperture_dir.mkdir(parents=True, exist_ok=True)

        with tqdm(
            date_ordered_files(gtfs_root),
            desc="GTFS files",
            bar_format=tqdm_bar_format,
            colour="#44AA99",
            leave=False,
            ) as inner_bar:
            for gtfs_path in inner_bar:
                inner_bar.set_postfix_str(f"{gtfs_path.stem}")

                try:
                    transport_network = r5py.TransportNetwork(
                        osm_pbf=output_pbf,
                        gtfs=gtfs_path,
                        elevation_model=out_folder / 'A/dem_bh.tiff',
                        # Default: elevation_cost_function=<ElevationCostFunction.TOBLER: 'TOBLER'>
                    )

                    travel_times = r5py.TravelTimeMatrix(
                        transport_network,
                        origins=data,
                        departure=_get_departure_date(gtfs_path),
                        departure_time_window=dt.timedelta(hours=2),
                        max_time=dt.timedelta(minutes=120),
                        max_public_transport_rides=2,
                        transport_modes=[
                            r5py.TransportMode.TRANSIT,
                            r5py.TransportMode.WALK,
                        ],
                        snap_to_network=np.ceil(
                            h3.average_hexagon_edge_length(
                                aperture,
                                unit="m"
                            )
                        ),
                    )

                    parquet_file = aperture_dir / f"{gtfs_path.stem}.parquet"
                    travel_times.to_parquet(parquet_file, index=False)

                except Exception as e:
                    tqdm.write(f"🤚 Skipping {gtfs_path.stem}: {e}")
                    continue
